# Training on Conversation Trajectories

This notebook loads the generated conversation trajectories from the simulator,
preprocesses them into training-ready formats, and fine-tunes a language model
on the persuasion/dialogue data.

## Data Structure
Each experiment result contains:
- **conversation**: Multi-turn dialogue between two personas (speaker, message, turn_index)
- **pre_survey / post_survey**: Opinion measurements before/after the conversation
- **changes**: Which survey answers shifted (measures persuasion effectiveness)
- **config**: Scenario metadata (adversarial mode, num_turns, model used)

In [ ]:
# Install dependencies (run once)
# !pip install transformers datasets torch peft accelerate trl bitsandbytes

In [ ]:
import json
import glob
from pathlib import Path
from typing import List, Dict

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

## 1. Load Conversation Trajectories

In [ ]:
def load_experiment_results(results_dir: str = "results/experiments") -> List[Dict]:
    """Load all experiment result JSON files."""
    experiments = []
    for filepath in sorted(glob.glob(f"{results_dir}/*.json")):
        with open(filepath) as f:
            data = json.load(f)
        for result in data.get("results", []):
            result["_source_file"] = Path(filepath).name
            experiments.append(result)
    return experiments


def load_conversation_logs(conv_dir: str = "results/conversations") -> List[Dict]:
    """Load standalone conversation result files."""
    conversations = []
    for filepath in sorted(glob.glob(f"{conv_dir}/conversation_results_*.json")):
        with open(filepath) as f:
            data = json.load(f)
        if "conversation_and_surveys" in data:
            conversations.append(data)
        elif "conversation" in data:
            conversations.append(data)
    return conversations


experiments = load_experiment_results()
conversations = load_conversation_logs()

print(f"Loaded {len(experiments)} experiment trajectories")
print(f"Loaded {len(conversations)} standalone conversations")

In [ ]:
# Inspect one trajectory
if experiments:
    ex = experiments[0]
    print(f"Source: {ex['_source_file']}")
    print(f"Adversarial: {ex['config'].get('adversarial', False)}")
    print(f"Turns: {len(ex['conversation'])}")
    print(f"Survey changes: {sum(1 for c in ex.get('changes', []) if c['changed'])} / {len(ex.get('changes', []))}")
    print(f"\nFirst 2 turns:")
    for turn in ex["conversation"][:2]:
        print(f"  [{turn['speaker']}]: {turn['message'][:100]}...")

## 2. Format Trajectories for Training

We support multiple training objectives:
- **Dialogue modeling**: Train on full conversations (next-turn prediction)
- **Persuasion-conditioned generation**: Condition on whether persuasion succeeded
- **Preference pairs**: Use successful vs unsuccessful persuasion for DPO/RLHF

In [ ]:
def format_as_chat_turns(conversation: List[Dict]) -> List[Dict]:
    """Convert conversation to chat-style messages for SFT."""
    messages = []
    for turn in conversation:
        role = "user" if turn["turn_index"] % 2 == 0 else "assistant"
        messages.append({
            "role": role,
            "content": f"[{turn['speaker']}]: {turn['message']}"
        })
    return messages


def format_as_single_text(conversation: List[Dict], include_metadata: bool = True) -> str:
    """Format conversation as a single training text with optional metadata prefix."""
    lines = []
    for turn in conversation:
        lines.append(f"{turn['speaker']}: {turn['message']}")
    return "\n\n".join(lines)


def format_persuasion_conditioned(
    experiment: Dict,
    target_speaker: str = "Bob",  # the persuader
) -> Dict:
    """Format with persuasion outcome as a conditioning signal.
    
    Returns a dict with:
      - text: the formatted training example
      - persuasion_score: number of survey answers that changed (0-4)
      - successful: whether any opinion shift occurred
    """
    changes = experiment.get("changes", [])
    num_changed = sum(1 for c in changes if c["changed"])
    successful = num_changed > 0
    
    # Build system prompt with persuasion context
    config = experiment.get("config", {})
    scenario = config.get("scenario_name", "unknown")
    
    prefix = f"[SCENARIO: {scenario}] [PERSUASION_OUTCOME: {'success' if successful else 'failure'}] [SHIFTS: {num_changed}/{len(changes)}]\n\n"
    
    conversation_text = format_as_single_text(experiment["conversation"])
    
    return {
        "text": prefix + conversation_text,
        "persuasion_score": num_changed,
        "successful": successful,
        "scenario": scenario,
        "num_turns": len(experiment["conversation"]),
    }


# Format all experiments
training_examples = []
for exp in experiments:
    example = format_persuasion_conditioned(exp)
    training_examples.append(example)

print(f"Created {len(training_examples)} training examples")
print(f"Successful persuasion: {sum(1 for e in training_examples if e['successful'])} / {len(training_examples)}")
print(f"Avg persuasion score: {sum(e['persuasion_score'] for e in training_examples) / max(len(training_examples), 1):.2f}")

In [ ]:
# Preview a formatted example
if training_examples:
    print(training_examples[0]["text"][:500])
    print("...")

## 3. Build HuggingFace Dataset

In [ ]:
dataset = Dataset.from_list(training_examples)
print(dataset)
print(f"\nColumns: {dataset.column_names}")
print(f"Example text length (chars): {len(dataset[0]['text'])}")

In [ ]:
# Train/val split
if len(dataset) > 3:
    split = dataset.train_test_split(test_size=0.2, seed=42)
    train_dataset = split["train"]
    eval_dataset = split["test"]
else:
    # Too few examples for a split — use all for training
    train_dataset = dataset
    eval_dataset = None

print(f"Train: {len(train_dataset)} examples")
print(f"Eval: {len(eval_dataset) if eval_dataset else 0} examples")

## 4. Model & Tokenizer Setup

Using QLoRA for efficient fine-tuning on Qwen3-4B.

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 2048

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Training

In [ ]:
OUTPUT_DIR = "./checkpoints/qwen-persuasion-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch" if eval_dataset else "no",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
)

In [ ]:
trainer.train()

In [ ]:
# Save the LoRA adapter
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"Model saved to {OUTPUT_DIR}/final")

## 6. Inference — Generate a Persuasive Response

In [ ]:
def generate_response(prompt: str, max_new_tokens: int = 256) -> str:
    """Generate a continuation from the fine-tuned model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated


# Test: condition on successful persuasion and generate Bob's next turn
test_prompt = """[SCENARIO: systems-vs-rl-theory-phd] [PERSUASION_OUTCOME: success] [SHIFTS: 3/4]

Alice: I'm pretty committed to doing a Systems PhD. There's something satisfying about building things that actually run at scale.

Bob:"""

print(generate_response(test_prompt))

## 7. (Optional) DPO — Preference Learning from Persuasion Outcomes

If you have multiple runs of the same scenario with different persuasion outcomes,
you can build preference pairs for Direct Preference Optimization.

In [ ]:
from itertools import combinations


def build_dpo_pairs(experiments: List[Dict]) -> List[Dict]:
    """Build preference pairs: higher persuasion_score = chosen, lower = rejected.
    
    Groups experiments by scenario, then pairs trajectories where one
    achieved more opinion shifts than the other.
    """
    # Group by scenario
    by_scenario = {}
    for exp in experiments:
        scenario = exp.get("config", {}).get("scenario_name", "unknown")
        by_scenario.setdefault(scenario, []).append(exp)
    
    pairs = []
    for scenario, exps in by_scenario.items():
        for a, b in combinations(exps, 2):
            score_a = sum(1 for c in a.get("changes", []) if c["changed"])
            score_b = sum(1 for c in b.get("changes", []) if c["changed"])
            
            if score_a == score_b:
                continue
            
            chosen = a if score_a > score_b else b
            rejected = b if score_a > score_b else a
            
            # Extract the persuader's turns only
            chosen_text = format_as_single_text(chosen["conversation"])
            rejected_text = format_as_single_text(rejected["conversation"])
            
            pairs.append({
                "prompt": f"[SCENARIO: {scenario}] Generate a persuasive conversation.",
                "chosen": chosen_text,
                "rejected": rejected_text,
            })
    
    return pairs


dpo_pairs = build_dpo_pairs(experiments)
print(f"Built {len(dpo_pairs)} DPO preference pairs")

if dpo_pairs:
    dpo_dataset = Dataset.from_list(dpo_pairs)
    print(dpo_dataset)

In [ ]:
# DPO training (requires trl >= 0.7)
# from trl import DPOTrainer, DPOConfig
#
# dpo_config = DPOConfig(
#     output_dir="./checkpoints/qwen-persuasion-dpo",
#     num_train_epochs=1,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,
#     learning_rate=5e-5,
#     beta=0.1,
#     bf16=True,
#     report_to="none",
# )
#
# dpo_trainer = DPOTrainer(
#     model=model,
#     ref_model=None,  # uses implicit reference with LoRA
#     train_dataset=dpo_dataset,
#     tokenizer=tokenizer,
#     args=dpo_config,
# )
# dpo_trainer.train()

## 8. Analysis — Persuasion Effectiveness by Scenario

In [ ]:
import statistics

# Aggregate stats per scenario
scenario_stats = {}
for exp in experiments:
    scenario = exp.get("config", {}).get("scenario_name", "unknown")
    num_changed = sum(1 for c in exp.get("changes", []) if c["changed"])
    total_qs = len(exp.get("changes", []))
    scenario_stats.setdefault(scenario, []).append(num_changed / max(total_qs, 1))

print(f"{'Scenario':<45} {'Runs':>5} {'Mean Shift':>11} {'Std':>6}")
print("-" * 70)
for scenario, scores in sorted(scenario_stats.items()):
    mean = statistics.mean(scores)
    std = statistics.stdev(scores) if len(scores) > 1 else 0.0
    print(f"{scenario:<45} {len(scores):>5} {mean:>10.2%} {std:>6.2%}")